In [1]:
from pathlib import Path

# Project and metadata paths
project_path = Path.cwd().parent
metadata_path = project_path / "data" / "raw" / "asl_citizen" / "metadata"

# Find all extracted metadata files
metadata_files = list(metadata_path.glob("*"))

print("Metadata folder:", metadata_path)
print("Total metadata files:", len(metadata_files))

for file in metadata_files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name} | {size_mb:.2f} MB")

Metadata folder: c:\Users\HP\OneDrive\Documents\NeuraSign\data\raw\asl_citizen\metadata
Total metadata files: 4
ASL_Citizen_splits_test.csv | 1.55 MB
ASL_Citizen_splits_train.csv | 1.91 MB
ASL_Citizen_splits_val.csv | 0.49 MB
ASL_Citizen_use.txt | 0.01 MB


In [2]:
import pandas as pd

train_file = metadata_path / "ASL_Citizen_splits_train.csv"
val_file = metadata_path / "ASL_Citizen_splits_val.csv"
test_file = metadata_path / "ASL_Citizen_splits_test.csv"

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)
test_df = pd.read_csv(test_file)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumn names:")
print(train_df.columns.tolist())

print("\nFirst 5 training records:")
display(train_df.head())

print("\nData types:")
print(train_df.dtypes)

Train shape: (40154, 4)
Validation shape: (10304, 4)
Test shape: (32941, 4)

Column names:
['Participant ID', 'Video file', 'Gloss', 'ASL-LEX Code']

First 5 training records:


,Participant ID,Video file,Gloss,ASL-LEX Code
0,P1,15890366051589533-APPLE.mp4,APPLE,A_03_054
1,P1,35618482303951104-IMPOSSIBLE.mp4,IMPOSSIBLE,B_01_032
2,P1,6958143575951994-PARK.mp4,PARK,E_03_028
3,P1,8006032738002744-SOCCER 2.mp4,SOCCER2,F_03_032
4,P1,37542279833186454-STINK.mp4,STINK,H_01_064



Data types:
Participant ID    str
Video file        str
Gloss             str
ASL-LEX Code      str
dtype: object


In [3]:
# Rename columns to simple names
new_columns = {
    "Participant ID": "participant_id",
    "Video file": "video_file",
    "Gloss": "gloss",
    "ASL-LEX Code": "asl_lex_code"
}

train_df = train_df.rename(columns=new_columns)
val_df = val_df.rename(columns=new_columns)
test_df = test_df.rename(columns=new_columns)

# Preserve official splits
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine metadata for inspection
all_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Remove accidental spaces from text columns
text_columns = [
    "participant_id",
    "video_file",
    "gloss",
    "asl_lex_code"
]

for column in text_columns:
    all_df[column] = all_df[column].astype("string").str.strip()

print("Total video records:", len(all_df))
print("\nSplit distribution:")
print(all_df["split"].value_counts())

print("\nMissing values:")
print(all_df.isna().sum())

print("\nDuplicate video filenames:")
print(all_df["video_file"].duplicated().sum())

print("\nUnique participants:", all_df["participant_id"].nunique())
print("Unique glosses:", all_df["gloss"].nunique())
print("Unique ASL-LEX classes:", all_df["asl_lex_code"].nunique())

display(all_df.head())

Total video records: 83399

Split distribution:
split
train         40154
test          32941
validation    10304
Name: count, dtype: int64

Missing values:
participant_id      0
video_file          0
gloss               0
asl_lex_code      183
split               0
dtype: int64

Duplicate video filenames:
0

Unique participants: 52
Unique glosses: 2731
Unique ASL-LEX classes: 2722


,participant_id,video_file,gloss,asl_lex_code,split
0,P1,15890366051589533-APPLE.mp4,APPLE,A_03_054,train
1,P1,35618482303951104-IMPOSSIBLE.mp4,IMPOSSIBLE,B_01_032,train
2,P1,6958143575951994-PARK.mp4,PARK,E_03_028,train
3,P1,8006032738002744-SOCCER 2.mp4,SOCCER2,F_03_032,train
4,P1,37542279833186454-STINK.mp4,STINK,H_01_064,train


In [4]:
from pathlib import Path
import zipfile

# Locate the ASL Citizen ZIP file
dataset_folder = (
    Path.home()
    / "OneDrive"
    / "Documents"
    / "NeuraSign_Datasets"
)

zip_files = [
    file for file in dataset_folder.iterdir()
    if file.is_file() and file.name.startswith("ASL_Citizen")
]

if not zip_files:
    raise FileNotFoundError("ASL Citizen ZIP file was not found.")

zip_path = zip_files[0]

print("ZIP file:", zip_path.name)
print("Reading complete video inventory...")

# Read the names of all videos without extracting them
with zipfile.ZipFile(zip_path, "r") as zip_file:
    video_entries = [
        item.filename
        for item in zip_file.infolist()
        if item.filename.lower().endswith(".mp4")
    ]

# Map each video filename to its complete path inside the ZIP
video_entry_map = {
    Path(entry).name: entry
    for entry in video_entries
}

# Check every CSV record against the ZIP
all_df["video_in_zip"] = all_df["video_file"].isin(video_entry_map)

total_metadata_videos = len(all_df)
total_zip_videos = len(video_entries)
matched_videos = int(all_df["video_in_zip"].sum())
missing_videos = total_metadata_videos - matched_videos

print("\nTotal metadata records:", total_metadata_videos)
print("Total MP4 videos in ZIP:", total_zip_videos)
print("Matched videos:", matched_videos)
print("Missing videos:", missing_videos)

# Display missing records, if any
if missing_videos > 0:
    display(
        all_df.loc[
            ~all_df["video_in_zip"],
            ["participant_id", "video_file", "gloss", "asl_lex_code", "split"]
        ].head(20)
    )
else:
    print("\nSuccess: Every metadata record has a corresponding video.")

ZIP file: ASL_Citizen.zip
Reading complete video inventory...

Total metadata records: 83399
Total MP4 videos in ZIP: 83399
Matched videos: 83399
Missing videos: 0

Success: Every metadata record has a corresponding video.


In [5]:
# Add the complete internal ZIP path for every video
all_df["zip_entry"] = all_df["video_file"].map(video_entry_map)

# Create the processed-data folder
processed_folder = (
    project_path
    / "data"
    / "processed"
    / "asl_citizen"
)

processed_folder.mkdir(parents=True, exist_ok=True)

# Save complete metadata
metadata_output = processed_folder / "asl_citizen_metadata_clean.csv"

all_df.to_csv(
    metadata_output,
    index=False
)

# Create ASL-LEX code to English gloss mapping
class_mapping = (
    all_df[
        ["asl_lex_code", "gloss"]
    ]
    .drop_duplicates()
    .sort_values(
        ["asl_lex_code", "gloss"]
    )
    .reset_index(drop=True)
)

mapping_output = processed_folder / "asl_citizen_class_mapping.csv"

class_mapping.to_csv(
    mapping_output,
    index=False
)

print("Complete metadata saved:", metadata_output)
print("Class mapping saved:", mapping_output)

print("\nSaved metadata records:", len(all_df))
print("Saved class mappings:", len(class_mapping))
print("Missing ZIP paths:", all_df["zip_entry"].isna().sum())

display(class_mapping.head(10))

Complete metadata saved: c:\Users\HP\OneDrive\Documents\NeuraSign\data\processed\asl_citizen\asl_citizen_metadata_clean.csv
Class mapping saved: c:\Users\HP\OneDrive\Documents\NeuraSign\data\processed\asl_citizen\asl_citizen_class_mapping.csv

Saved metadata records: 83399
Saved class mappings: 2731
Missing ZIP paths: 0


,asl_lex_code,gloss
0,A_01_002,TREE
1,A_01_003,NIGHT1
2,A_01_004,HAMBURGER
3,A_01_005,NEPHEW
4,A_01_008,CASTLE1
5,A_01_009,HUMBLE
6,A_01_010,CUP
7,A_01_011,ENGLISH
8,A_01_012,EYEGLASSES
9,A_01_013,TIE1


In [6]:
from pathlib import Path
import zipfile
import tempfile

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Project paths
project_path = Path.cwd().parent

processed_folder = (
    project_path
    / "data"
    / "processed"
    / "asl_citizen"
)

metadata_file = (
    processed_folder
    / "asl_citizen_metadata_clean.csv"
)

# Load the complete cleaned metadata
all_df = pd.read_csv(metadata_file)

# Locate the original ASL Citizen ZIP
dataset_folder = (
    Path.home()
    / "OneDrive"
    / "Documents"
    / "NeuraSign_Datasets"
)

zip_path = next(
    file
    for file in dataset_folder.iterdir()
    if file.is_file()
    and file.name.startswith("ASL_Citizen")
)

print("Metadata records:", len(all_df))
print("ZIP file:", zip_path.name)
print("OpenCV version:", cv2.__version__)
print("MediaPipe version:", mp.__version__)
print("Environment restored successfully.")

Metadata records: 83399
ZIP file: ASL_Citizen.zip
OpenCV version: 5.0.0
MediaPipe version: 1.0.1
Environment restored successfully.


In [7]:
# Check MediaPipe landmark models
print("MediaPipe solutions available:", hasattr(mp, "solutions"))

if hasattr(mp, "solutions"):
    print("Hands model:", hasattr(mp.solutions, "hands"))
    print("Pose model:", hasattr(mp.solutions, "pose"))
    print("Holistic model:", hasattr(mp.solutions, "holistic"))
else:
    print("Legacy MediaPipe Solutions API is not available.")
    

MediaPipe solutions available: False
Legacy MediaPipe Solutions API is not available.


In [1]:
import cv2
import mediapipe as mp
import numpy as np

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Solutions available:", hasattr(mp, "solutions"))
print("Holistic available:", hasattr(mp.solutions, "holistic"))

NumPy: 1.26.4
OpenCV: 4.11.0
MediaPipe: 0.10.21
Solutions available: True
Holistic available: True


In [2]:
from pathlib import Path
import zipfile
import tempfile
import shutil

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

# Restore project paths
project_path = Path.cwd().parent

metadata_file = (
    project_path
    / "data"
    / "processed"
    / "asl_citizen"
    / "asl_citizen_metadata_clean.csv"
)

all_df = pd.read_csv(metadata_file)

dataset_folder = (
    Path.home()
    / "OneDrive"
    / "Documents"
    / "NeuraSign_Datasets"
)

zip_path = next(
    file for file in dataset_folder.iterdir()
    if file.is_file()
    and file.name.startswith("ASL_Citizen")
)

print("Metadata records:", len(all_df))
print("MediaPipe version:", mp.__version__)
print("Setup restored successfully.")

Metadata records: 83399
MediaPipe version: 0.10.21
Setup restored successfully.


In [3]:
mp_holistic = mp.solutions.holistic


def landmarks_to_array(landmarks, total_points, dimensions):
    if landmarks is None:
        return np.zeros(
            total_points * dimensions,
            dtype=np.float32
        )

    values = []

    for point in landmarks.landmark:
        values.extend([
            point.x,
            point.y,
            point.z
        ])

        if dimensions == 4:
            values.append(point.visibility)

    return np.array(
        values,
        dtype=np.float32
    )


def extract_video_features(video_path, sample_frames=20):
    frame_features = []

    video = cv2.VideoCapture(str(video_path))

    total_frames = int(
        video.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        video.get(cv2.CAP_PROP_FPS)
    )

    width = int(
        video.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        video.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    if total_frames <= 0:
        video.release()
        raise ValueError(
            "The video contains no readable frames."
        )

    selected_frames = np.linspace(
        0,
        total_frames - 1,
        sample_frames,
        dtype=int
    )

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        for frame_number in selected_frames:
            video.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_number)
            )

            success, frame = video.read()

            if not success:
                continue

            rgb_frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            results = holistic.process(rgb_frame)

            pose = landmarks_to_array(
                results.pose_landmarks,
                total_points=33,
                dimensions=4
            )

            left_hand = landmarks_to_array(
                results.left_hand_landmarks,
                total_points=21,
                dimensions=3
            )

            right_hand = landmarks_to_array(
                results.right_hand_landmarks,
                total_points=21,
                dimensions=3
            )

            combined = np.concatenate([
                pose,
                left_hand,
                right_hand
            ])

            frame_features.append(combined)

    video.release()

    if not frame_features:
        raise ValueError(
            "No frames could be processed."
        )

    sequence = np.vstack(frame_features)

    mean_features = sequence.mean(axis=0)
    standard_deviation = sequence.std(axis=0)

    if len(sequence) > 1:
        motion_features = np.abs(
            np.diff(sequence, axis=0)
        ).mean(axis=0)
    else:
        motion_features = np.zeros(
            sequence.shape[1],
            dtype=np.float32
        )

    id3_features = np.concatenate([
        mean_features,
        standard_deviation,
        motion_features
    ])

    duration = (
        total_frames / fps
        if fps > 0
        else 0
    )

    video_info = {
        "total_frames": total_frames,
        "processed_frames": len(sequence),
        "fps": fps,
        "width": width,
        "height": height,
        "duration_seconds": duration,
        "feature_count": len(id3_features),
        "non_zero_features": int(
            np.count_nonzero(id3_features)
        )
    }

    return sequence, id3_features, video_info

In [4]:
# Select the first video record
test_record = all_df.iloc[0]

# Create a temporary MP4 file
temporary_file = tempfile.NamedTemporaryFile(
    suffix=".mp4",
    delete=False
)

temporary_path = Path(
    temporary_file.name
)

temporary_file.close()

try:
    # Extract only one video temporarily from the ZIP
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        with zip_file.open(
            test_record["zip_entry"]
        ) as source:

            with open(
                temporary_path,
                "wb"
            ) as destination:

                shutil.copyfileobj(
                    source,
                    destination
                )

    # Extract landmarks and numerical features
    sequence, id3_features, video_info = (
        extract_video_features(
            temporary_path,
            sample_frames=20
        )
    )

    print("Video:", test_record["video_file"])
    print("Gloss:", test_record["gloss"])
    print(
        "ASL-LEX code:",
        test_record["asl_lex_code"]
    )

    print(
        "Sequence shape:",
        sequence.shape
    )

    print(
        "ID3 feature shape:",
        id3_features.shape
    )

    print("\nVideo information:")

    for key, value in video_info.items():
        print(f"{key}: {value}")

finally:
    # Delete the temporary video
    temporary_path.unlink(
        missing_ok=True
    )

Video: 15890366051589533-APPLE.mp4
Gloss: APPLE
ASL-LEX code: A_03_054
Sequence shape: (20, 258)
ID3 feature shape: (774,)

Video information:
total_frames: 71
processed_frames: 20
fps: 29.91993257479983
width: 640
height: 480
duration_seconds: 2.373
feature_count: 774
non_zero_features: 585


In [5]:
import time

test_results = []
start_time = time.time()

# Test the complete pipeline on five videos
for index in range(5):
    record = all_df.iloc[index]

    temporary_file = tempfile.NamedTemporaryFile(
        suffix=".mp4",
        delete=False
    )

    temporary_path = Path(temporary_file.name)
    temporary_file.close()

    try:
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            with zip_file.open(record["zip_entry"]) as source:
                with open(temporary_path, "wb") as destination:
                    shutil.copyfileobj(source, destination)

        sequence, features, information = extract_video_features(
            temporary_path,
            sample_frames=20
        )

        test_results.append({
            "video_file": record["video_file"],
            "gloss": record["gloss"],
            "asl_lex_code": record["asl_lex_code"],
            "processed_frames": information["processed_frames"],
            "feature_count": information["feature_count"],
            "non_zero_features": information["non_zero_features"],
            "status": "success"
        })

        print(
            f"{index + 1}/5 completed:",
            record["gloss"]
        )

    except Exception as error:
        test_results.append({
            "video_file": record["video_file"],
            "gloss": record["gloss"],
            "asl_lex_code": record["asl_lex_code"],
            "processed_frames": 0,
            "feature_count": 0,
            "non_zero_features": 0,
            "status": f"failed: {error}"
        })

        print(
            f"{index + 1}/5 failed:",
            error
        )

    finally:
        temporary_path.unlink(missing_ok=True)

elapsed_seconds = time.time() - start_time
average_seconds = elapsed_seconds / 5
estimated_hours = (
    average_seconds * len(all_df)
) / 3600

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

print(f"\nTest time: {elapsed_seconds:.2f} seconds")
print(f"Average per video: {average_seconds:.2f} seconds")
print(
    f"Estimated complete processing time: "
    f"{estimated_hours:.2f} hours"
)

1/5 completed: APPLE
2/5 completed: IMPOSSIBLE
3/5 completed: PARK
4/5 completed: SOCCER2
5/5 completed: STINK


,video_file,gloss,asl_lex_code,processed_frames,feature_count,non_zero_features,status
0,15890366051589533-APPLE.mp4,APPLE,A_03_054,20,774,585,success
1,35618482303951104-IMPOSSIBLE.mp4,IMPOSSIBLE,B_01_032,20,774,774,success
2,6958143575951994-PARK.mp4,PARK,E_03_028,20,774,774,success
3,8006032738002744-SOCCER 2.mp4,SOCCER2,F_03_032,20,774,774,success
4,37542279833186454-STINK.mp4,STINK,H_01_064,20,774,585,success



Test time: 9.48 seconds
Average per video: 1.90 seconds
Estimated complete processing time: 43.94 hours


In [6]:
def extract_video_features(
    video_path,
    holistic_model,
    sample_frames=20
):
    frame_features = []

    video = cv2.VideoCapture(str(video_path))

    total_frames = int(
        video.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        video.get(cv2.CAP_PROP_FPS)
    )

    width = int(
        video.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        video.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    if total_frames <= 0:
        video.release()

        raise ValueError(
            "The video contains no readable frames."
        )

    selected_frames = np.linspace(
        0,
        total_frames - 1,
        sample_frames,
        dtype=int
    )

    for frame_number in selected_frames:
        video.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_number)
        )

        success, frame = video.read()

        if not success:
            continue

        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        results = holistic_model.process(
            rgb_frame
        )

        pose = landmarks_to_array(
            results.pose_landmarks,
            total_points=33,
            dimensions=4
        )

        left_hand = landmarks_to_array(
            results.left_hand_landmarks,
            total_points=21,
            dimensions=3
        )

        right_hand = landmarks_to_array(
            results.right_hand_landmarks,
            total_points=21,
            dimensions=3
        )

        combined = np.concatenate([
            pose,
            left_hand,
            right_hand
        ])

        frame_features.append(combined)

    video.release()

    if not frame_features:
        raise ValueError(
            "No frames could be processed."
        )

    sequence = np.vstack(frame_features)

    # Pad unreadable frames with zeros
    if len(sequence) < sample_frames:
        missing_frames = (
            sample_frames - len(sequence)
        )

        padding = np.zeros(
            (
                missing_frames,
                sequence.shape[1]
            ),
            dtype=np.float32
        )

        sequence = np.vstack([
            sequence,
            padding
        ])

    mean_features = sequence.mean(axis=0)

    standard_deviation = sequence.std(axis=0)

    motion_features = np.abs(
        np.diff(sequence, axis=0)
    ).mean(axis=0)

    id3_features = np.concatenate([
        mean_features,
        standard_deviation,
        motion_features
    ]).astype(np.float32)

    duration = (
        total_frames / fps
        if fps > 0
        else 0
    )

    video_info = {
        "total_frames": total_frames,
        "processed_frames": len(frame_features),
        "fps": fps,
        "width": width,
        "height": height,
        "duration_seconds": duration,
        "feature_count": len(id3_features),
        "non_zero_features": int(
            np.count_nonzero(id3_features)
        )
    }

    return (
        sequence.astype(np.float32),
        id3_features,
        video_info
    )


print("Optimized extraction function is ready.")

Optimized extraction function is ready.


In [9]:
import math
import time

# Processing settings
BATCH_SIZE = 250
SAMPLE_FRAMES = 20

feature_folder = (
    project_path
    / "data"
    / "processed"
    / "asl_citizen"
    / "feature_shards"
)

feature_folder.mkdir(
    parents=True,
    exist_ok=True
)

total_records = len(all_df)

total_batches = math.ceil(
    total_records / BATCH_SIZE
)

print("Total videos:", total_records)
print("Batch size:", BATCH_SIZE)
print("Total batches:", total_batches)
print("Output folder:", feature_folder)

overall_start_time = time.time()

# Open the ZIP and MediaPipe model only once
with zipfile.ZipFile(zip_path, "r") as zip_file:

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic_model:

        with tempfile.TemporaryDirectory() as temp_directory:
            temp_video_path = (
                Path(temp_directory)
                / "current_video.mp4"
            )

            for batch_number in range(total_batches):
                batch_start = (
                    batch_number * BATCH_SIZE
                )

                batch_end = min(
                    batch_start + BATCH_SIZE,
                    total_records
                )

                shard_path = (
                    feature_folder
                    / f"batch_{batch_number:04d}.npz"
                )

                # Skip batches already processed
                if shard_path.exists():
                    print(
                        f"Batch {batch_number + 1}/"
                        f"{total_batches} already exists. "
                        f"Skipping."
                    )
                    continue

                batch_df = all_df.iloc[
                    batch_start:batch_end
                ]

                batch_sequences = []
                batch_id3_features = []

                video_files = []
                participant_ids = []
                glosses = []
                asl_lex_codes = []
                splits = []

                success_flags = []
                error_messages = []

                total_frame_values = []
                processed_frame_values = []
                fps_values = []
                width_values = []
                height_values = []
                duration_values = []
                non_zero_values = []

                batch_time = time.time()

                for row_number, row in tqdm(
                    batch_df.iterrows(),
                    total=len(batch_df),
                    desc=(
                        f"Batch {batch_number + 1}/"
                        f"{total_batches}"
                    )
                ):
                    success = True
                    error_message = ""

                    try:
                        # Extract one video temporarily
                        with zip_file.open(
                            row["zip_entry"]
                        ) as source:

                            with open(
                                temp_video_path,
                                "wb"
                            ) as destination:

                                shutil.copyfileobj(
                                    source,
                                    destination
                                )

                        sequence, id3_vector, info = (
                            extract_video_features(
                                temp_video_path,
                                holistic_model,
                                sample_frames=SAMPLE_FRAMES
                            )
                        )

                    except Exception as error:
                        success = False
                        error_message = str(error)

                        # Preserve the record using zero features
                        sequence = np.zeros(
                            (
                                SAMPLE_FRAMES,
                                258
                            ),
                            dtype=np.float32
                        )

                        id3_vector = np.zeros(
                            774,
                            dtype=np.float32
                        )

                        info = {
                            "total_frames": 0,
                            "processed_frames": 0,
                            "fps": 0,
                            "width": 0,
                            "height": 0,
                            "duration_seconds": 0,
                            "non_zero_features": 0
                        }

                    batch_sequences.append(sequence)
                    batch_id3_features.append(id3_vector)

                    video_files.append(
                        str(row["video_file"])
                    )

                    participant_ids.append(
                        str(row["participant_id"])
                    )

                    glosses.append(
                        str(row["gloss"])
                    )

                    asl_lex_codes.append(
                        str(row["asl_lex_code"])
                    )

                    splits.append(
                        str(row["split"])
                    )

                    success_flags.append(success)
                    error_messages.append(error_message)

                    total_frame_values.append(
                        info["total_frames"]
                    )

                    processed_frame_values.append(
                        info["processed_frames"]
                    )

                    fps_values.append(
                        info["fps"]
                    )

                    width_values.append(
                        info["width"]
                    )

                    height_values.append(
                        info["height"]
                    )

                    duration_values.append(
                        info["duration_seconds"]
                    )

                    non_zero_values.append(
                        info["non_zero_features"]
                    )

                # Convert batch lists into arrays
                sequences_array = np.stack(
                    batch_sequences
                ).astype(np.float32)

                id3_array = np.stack(
                    batch_id3_features
                ).astype(np.float32)

                # Save atomically to protect completed batches
                temporary_shard = (
                    feature_folder
                    / f"batch_{batch_number:04d}.tmp.npz"
                )

                np.savez_compressed(
                    temporary_shard,
                    sequences=sequences_array,
                    id3_features=id3_array,
                    video_file=np.array(video_files),
                    participant_id=np.array(participant_ids),
                    gloss=np.array(glosses),
                    asl_lex_code=np.array(asl_lex_codes),
                    split=np.array(splits),
                    success=np.array(
                        success_flags,
                        dtype=bool
                    ),
                    error_message=np.array(error_messages),
                    total_frames=np.array(total_frame_values),
                    processed_frames=np.array(
                        processed_frame_values
                    ),
                    fps=np.array(fps_values),
                    width=np.array(width_values),
                    height=np.array(height_values),
                    duration_seconds=np.array(
                        duration_values
                    ),
                    non_zero_features=np.array(
                        non_zero_values
                    )
                )

                temporary_shard.replace(shard_path)

                successful_count = int(
                    np.sum(success_flags)
                )

                failed_count = (
                    len(success_flags)
                    - successful_count
                )

                batch_minutes = (
                    time.time() - batch_time
                ) / 60

                print(
                    f"\nSaved batch "
                    f"{batch_number + 1}/"
                    f"{total_batches}"
                )

                print(
                    f"Records: "
                    f"{batch_start} to "
                    f"{batch_end - 1}"
                )

                print(
                    f"Successful: "
                    f"{successful_count}"
                )

                print(
                    f"Failed: {failed_count}"
                )

                print(
                    f"Batch time: "
                    f"{batch_minutes:.2f} minutes"
                )

total_hours = (
    time.time() - overall_start_time
) / 3600

print("\nComplete ASL Citizen processing finished.")
print(f"Total runtime: {total_hours:.2f} hours")

Total videos: 83399
Batch size: 250
Total batches: 334
Output folder: c:\Users\HP\OneDrive\Documents\NeuraSign\data\processed\asl_citizen\feature_shards


Batch 1/334:   0%|          | 0/250 [00:00<?, ?it/s]


Saved batch 1/334
Records: 0 to 249
Successful: 250
Failed: 0
Batch time: 7.46 minutes


Batch 2/334:   0%|          | 0/250 [00:00<?, ?it/s]


Saved batch 2/334
Records: 250 to 499
Successful: 250
Failed: 0
Batch time: 6.67 minutes


...


Saved batch 10/334
Records: 2250 to 2499
Successful: 250
Failed: 0
Batch time: 21.08 minutes


Batch 11/334:   0%|          | 0/250 [00:00<?, ?it/s]


Saved batch 11/334
Records: 2500 to 2749
Successful: 250
Failed: 0
Batch time: 34.62 minutes


Batch 12/334:   0%|          | 0/250 [00:00<?, ?it/s]

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\HP\\AppData\\Local\\Temp\\tmpmi_v45iz\\current_video.mp4'

In [10]:
def extract_fast_video_features(video_path, sample_frames=5):
    video = cv2.VideoCapture(str(video_path))

    total_frames = int(
        video.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        video.get(cv2.CAP_PROP_FPS)
    )

    width = int(
        video.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        video.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    if total_frames <= 0:
        video.release()
        raise ValueError("Video has no readable frames.")

    duration = (
        total_frames / fps
        if fps > 0
        else 0
    )

    aspect_ratio = (
        width / height
        if height > 0
        else 0
    )

    selected_frames = np.linspace(
        0,
        total_frames - 1,
        sample_frames,
        dtype=int
    )

    brightness_values = []
    contrast_values = []
    edge_values = []
    motion_values = []

    previous_frame = None
    readable_frames = 0

    for frame_number in selected_frames:
        video.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_number)
        )

        success, frame = video.read()

        if not success:
            continue

        readable_frames += 1

        # Resize for faster processing
        frame = cv2.resize(
            frame,
            (160, 120)
        )

        gray_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        brightness_values.append(
            float(gray_frame.mean())
        )

        contrast_values.append(
            float(gray_frame.std())
        )

        edges = cv2.Canny(
            gray_frame,
            100,
            200
        )

        edge_values.append(
            float(
                np.count_nonzero(edges)
                / edges.size
            )
        )

        if previous_frame is not None:
            frame_difference = cv2.absdiff(
                gray_frame,
                previous_frame
            )

            motion_values.append(
                float(frame_difference.mean())
            )

        previous_frame = gray_frame

    video.release()

    if readable_frames == 0:
        raise ValueError(
            "No selected frames could be read."
        )

    features = np.array([
        total_frames,
        fps,
        duration,
        width,
        height,
        aspect_ratio,
        np.mean(brightness_values),
        np.std(brightness_values),
        np.mean(contrast_values),
        np.mean(edge_values),
        np.mean(motion_values)
        if motion_values else 0
    ], dtype=np.float32)

    return features


feature_names = [
    "total_frames",
    "fps",
    "duration_seconds",
    "width",
    "height",
    "aspect_ratio",
    "mean_brightness",
    "brightness_variation",
    "mean_contrast",
    "edge_density",
    "motion_intensity"
]

print("Fast OpenCV feature function is ready.")
print("Number of features:", len(feature_names))

Fast OpenCV feature function is ready.
Number of features: 11


In [11]:
import time
from tqdm.auto import tqdm

test_feature_rows = []
test_start_time = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with tempfile.TemporaryDirectory() as temp_directory:
        temp_video_path = (
            Path(temp_directory)
            / "test_video.mp4"
        )

        for index in tqdm(
            range(20),
            desc="Testing fast extraction"
        ):
            record = all_df.iloc[index]

            try:
                # Extract one video temporarily
                with zip_file.open(
                    record["zip_entry"]
                ) as source:

                    with open(
                        temp_video_path,
                        "wb"
                    ) as destination:

                        shutil.copyfileobj(
                            source,
                            destination
                        )

                features = extract_fast_video_features(
                    temp_video_path,
                    sample_frames=5
                )

                test_feature_rows.append(features)

            except Exception as error:
                print(
                    f"Failed video {index}: {error}"
                )

elapsed_seconds = time.time() - test_start_time

processed_count = len(test_feature_rows)

average_seconds = (
    elapsed_seconds / processed_count
    if processed_count > 0
    else 0
)

estimated_hours = (
    average_seconds * len(all_df)
) / 3600

print("\nSuccessfully processed:", processed_count)
print(f"Test time: {elapsed_seconds:.2f} seconds")
print(f"Average per video: {average_seconds:.3f} seconds")
print(
    f"Estimated complete time: "
    f"{estimated_hours:.2f} hours"
)

if processed_count > 0:
    test_features_df = pd.DataFrame(
        test_feature_rows,
        columns=feature_names
    )

    display(test_features_df.head())

Testing fast extraction:   0%|          | 0/20 [00:00<?, ?it/s]


Successfully processed: 20
Test time: 8.68 seconds
Average per video: 0.434 seconds
Estimated complete time: 10.05 hours


,total_frames,fps,duration_seconds,width,height,aspect_ratio,mean_brightness,brightness_variation,mean_contrast,edge_density,motion_intensity
0,71.0,29.919933,2.373,640.0,480.0,1.333333,155.203171,0.908351,54.329857,0.091115,9.905925
1,76.0,29.968454,2.536,640.0,480.0,1.333333,154.512024,5.165944,53.633423,0.098698,15.588763
2,87.0,30.000000,2.900,640.0,480.0,1.333333,129.664093,52.662025,52.912628,0.058021,38.841759
3,112.0,30.002678,3.733,640.0,480.0,1.333333,156.401047,1.888887,63.354954,0.072177,10.262565
4,87.0,30.000000,2.900,640.0,480.0,1.333333,157.299210,0.337837,65.059517,0.072573,5.686823


In [12]:
import math
import time

BATCH_SIZE = 500
SAMPLE_FRAMES = 5

output_folder = (
    project_path
    / "data"
    / "processed"
    / "asl_citizen"
    / "opencv_feature_batches"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

total_records = len(all_df)

total_batches = math.ceil(
    total_records / BATCH_SIZE
)

print("Total videos:", total_records)
print("Batch size:", BATCH_SIZE)
print("Total batches:", total_batches)
print("Features per video:", len(feature_names))

overall_start_time = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with tempfile.TemporaryDirectory() as temp_directory:

        temp_video_path = (
            Path(temp_directory)
            / "current_video.mp4"
        )

        for batch_number in range(total_batches):
            batch_start = (
                batch_number * BATCH_SIZE
            )

            batch_end = min(
                batch_start + BATCH_SIZE,
                total_records
            )

            batch_output = (
                output_folder
                / f"batch_{batch_number:04d}.csv"
            )

            # Skip completed batches
            if batch_output.exists():
                print(
                    f"Batch {batch_number + 1}/"
                    f"{total_batches} already exists. "
                    f"Skipping."
                )
                continue

            batch_df = all_df.iloc[
                batch_start:batch_end
            ]

            batch_records = []
            batch_start_time = time.time()

            for _, row in tqdm(
                batch_df.iterrows(),
                total=len(batch_df),
                desc=(
                    f"Batch {batch_number + 1}/"
                    f"{total_batches}"
                )
            ):
                success = True
                error_message = ""

                try:
                    with zip_file.open(
                        row["zip_entry"]
                    ) as source:

                        with open(
                            temp_video_path,
                            "wb"
                        ) as destination:

                            shutil.copyfileobj(
                                source,
                                destination
                            )

                    features = (
                        extract_fast_video_features(
                            temp_video_path,
                            sample_frames=SAMPLE_FRAMES
                        )
                    )

                except Exception as error:
                    success = False
                    error_message = str(error)

                    # Keep missing features as NaN
                    features = np.full(
                        len(feature_names),
                        np.nan,
                        dtype=np.float32
                    )

                output_record = {
                    "participant_id": row["participant_id"],
                    "video_file": row["video_file"],
                    "gloss": row["gloss"],
                    "asl_lex_code": row["asl_lex_code"],
                    "split": row["split"],
                    "success": success,
                    "error_message": error_message
                }

                for feature_name, value in zip(
                    feature_names,
                    features
                ):
                    output_record[feature_name] = value

                batch_records.append(output_record)

            batch_result = pd.DataFrame(
                batch_records
            )

            temporary_output = (
                output_folder
                / f"batch_{batch_number:04d}.tmp.csv"
            )

            batch_result.to_csv(
                temporary_output,
                index=False
            )

            temporary_output.replace(
                batch_output
            )

            successful_count = int(
                batch_result["success"].sum()
            )

            failed_count = (
                len(batch_result)
                - successful_count
            )

            batch_minutes = (
                time.time() - batch_start_time
            ) / 60

            print(
                f"\nSaved batch "
                f"{batch_number + 1}/"
                f"{total_batches}"
            )

            print(
                f"Successful: {successful_count}"
            )

            print(
                f"Failed: {failed_count}"
            )

            print(
                f"Batch time: "
                f"{batch_minutes:.2f} minutes"
            )

total_hours = (
    time.time() - overall_start_time
) / 3600

print("\nAll OpenCV feature batches completed.")
print(f"Total runtime: {total_hours:.2f} hours")

Total videos: 83399
Batch size: 500
Total batches: 167
Features per video: 11


Batch 1/167:   0%|          | 0/500 [00:00<?, ?it/s]


Saved batch 1/167
Successful: 500
Failed: 0
Batch time: 3.32 minutes


Batch 2/167:   0%|          | 0/500 [00:00<?, ?it/s]


Saved batch 2/167
Successful: 500
Failed: 0
Batch time: 2.65 minutes


...


Saved batch 5/167
Successful: 500
Failed: 0
Batch time: 4.09 minutes


Batch 6/167:   0%|          | 0/500 [00:00<?, ?it/s]


Saved batch 6/167
Successful: 500
Failed: 0
Batch time: 4.70 minutes


Batch 7/167:   0%|          | 0/500 [00:00<?, ?it/s]

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\HP\\AppData\\Local\\Temp\\tmp9zhpbs0g\\current_video.mp4'

In [8]:
from tqdm.auto import tqdm

print("Progress bar imported successfully.")

Progress bar imported successfully.
